# Baseline training on Colab

Runs the code in `src/` on a Colab GPU. Use **Runtime → Change runtime type → GPU**.

`data/`, `weights/` and `results/` are kept on Google Drive, so downloads, embeddings and trained models survive between sessions.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo. If it's private, use https://<token>@github.com/... with a GitHub personal access token.
REPO = "https://github.com/aaronteork/CS5284_GML.git"
BRANCH = "feature/baseline"

!git clone -q -b {BRANCH} {REPO} /content/CS5284_GML || git -C /content/CS5284_GML pull -q
%cd /content/CS5284_GML

In [ ]:
# Keep data, weights and results on Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/cs5284_project")

for name in ["data", "weights", "results"]:
    target = DRIVE_DIR / name
    target.mkdir(parents=True, exist_ok=True)
    link = Path(name)
    if not link.is_symlink():
        !rm -rf {link}
        link.symlink_to(target)
    print(f"{name}/ -> {target}")

In [ ]:
!pip install -q -r requirements.txt

## Node2Vec embeddings (GraphSAGE only)

Skipped if `data/embeddings/node2vec/<state>_128.npy` already exists.

In [ ]:
!python -m src.data.node2vec --state DE MD

In [ ]:
# Sanity check: connected intersections should have more similar embeddings than random pairs
import numpy as np
import torch

for state in ["DE", "MD"]:
    emb = np.load(f"data/embeddings/node2vec/{state}_128.npy")
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    src, dst = torch.load(f"data/{state}/adj_matrix.pt", weights_only=False).coalesce().indices().numpy()

    rng = np.random.default_rng(0)
    edges = rng.choice(len(src), 5000)
    others = rng.integers(0, len(emb), 5000)
    neighbour = (emb[src[edges]] * emb[dst[edges]]).sum(1).mean()
    random = (emb[src[edges]] * emb[others]).sum(1).mean()
    print(f"{state}: neighbour cosine {neighbour:.3f} vs random {random:.3f}")
    assert neighbour > random + 0.2, f"{state} embeddings look untrained"

## Train

Each run trains every seed, keeps the best validation epoch in `weights/<run>/`, and writes `results/<run>/metrics.json`.

In [ ]:
STATE = "DE"
SEEDS = "0 1 2"
NODE2VEC = f"data/embeddings/node2vec/{STATE}_128.npy"

!python -m src.model.train --state {STATE} --model mlp --task classification --seeds {SEEDS}
!python -m src.model.train --state {STATE} --model mlp --task regression --seeds {SEEDS}
!python -m src.model.train --state {STATE} --model graphsage --task classification --node-features {NODE2VEC} --seeds {SEEDS}
!python -m src.model.train --state {STATE} --model graphsage --task regression --node-features {NODE2VEC} --seeds {SEEDS}

## Compare with the paper

In [ ]:
import json
import pandas as pd

# Paper Table 2 (test split, mean over 3 seeds)
PAPER = {
    ("DE", "mlp"): {"AUROC": 0.755, "MAE": 1.4},
    ("DE", "graphsage"): {"AUROC": 0.876, "MAE": 0.3},
    ("MD", "mlp"): {"AUROC": 0.741, "MAE": 0.4},
    ("MD", "graphsage"): {"AUROC": 0.876, "MAE": 0.4},
}

rows = []
for path in sorted(Path("results").glob("*/metrics.json")):
    result = json.load(open(path))
    cfg, test = result["config"], result["summary"]["test"]
    metric = "AUROC" if cfg["task"] == "classification" else "MAE"
    row = {"run": cfg["run_name"], "state": cfg["state"], "model": cfg["model"], "metric": metric}
    for protocol in ["all", "balanced"]:
        m = test[protocol][metric]
        row[protocol] = f"{m['mean']:.3f} ± {m['std']:.3f}"
    row["paper"] = PAPER.get((cfg["state"], cfg["model"]), {}).get(metric)
    rows.append(row)

pd.DataFrame(rows)